In [ ]:
import os
import sys
import subprocess
import gc
import warnings
import matplotlib.pyplot as plt
import numpy as np
import torch
import soundfile as sf
from google.colab import files

# Suppress warnings for cleaner output
warnings.filterwarnings("ignore")

def install_dependencies():
    """Checks and installs necessary system and python libraries."""
    print("Initializing environment and installing dependencies...")
    subprocess.run(["apt-get", "update", "-y"], stdout=subprocess.DEVNULL)
    subprocess.run(["apt-get", "install", "-y", "ffmpeg", "rubberband-cli"], stdout=subprocess.DEVNULL)

    packages = [
        "xformers", "av", "torchdiffeq", "torchmetrics", "antlr4-python3-runtime",
        "einops", "flashy>=0.0.1", "hydra-core>=1.1", "hydra-colorlog", "omegaconf",
        "sentencepiece", "spacy", "num2words", "tqdm", "transformers", "huggingface_hub",
        "encodec", "protobuf", "demucs", "librosa", "soundfile", "pedalboard",
        "pyrubberband", "scipy", "numpy", "matplotlib"
    ]
    subprocess.run([sys.executable, "-m", "pip", "install"] + packages, stdout=subprocess.DEVNULL)

    # Install AudioCraft source
    subprocess.run([sys.executable, "-m", "pip", "install", "git+https://github.com/facebookresearch/audiocraft.git", "--no-deps"], stdout=subprocess.DEVNULL)

# Dependency Verification
try:
    import audiocraft
    import pyrubberband
    import librosa
    from audiocraft.models import MusicGen
    from pedalboard import Pedalboard, Compressor, Limiter, HighpassFilter, Reverb
except ImportError:
    install_dependencies()
    import librosa
    import pyrubberband
    from audiocraft.models import MusicGen
    from pedalboard import Pedalboard, Compressor, Limiter, HighpassFilter, Reverb

# --- CORE PROCESSING FUNCTIONS ---

def generate_context_prompt(user_input, key_detected="C"):
    """
    Analyzes user input to generate context-aware prompts for the AI model.
    Handles specific genre mapping for religious/worship contexts.
    """
    u = user_input.lower()

    # Context Mapping
    if any(x in u for x in ["jesus", "gospel", "worship", "god"]):
        print("Style Detected: Gospel/Worship")
        return f"Cinematic gospel worship song, grand piano, slow emotional orchestral strings, ambient pads, divine atmosphere, Key {key_detected}, reverb"

    elif any(x in u for x in ["sad", "ballad", "emotional"]):
        print("Style Detected: Emotional Ballad")
        return f"Sad emotional ballad, acoustic piano, soft violins, melancholic atmosphere, slow tempo, minimal drums, Key {key_detected}"

    elif "rock" in u:
        print("Style Detected: Rock")
        return f"Soft rock ballad, electric guitar, bass, acoustic drums, emotional, Key {key_detected}"

    else:
        print(f"Style Detected: Cinematic {user_input}")
        return f"{user_input} song, cinematic, high fidelity, master quality, ambient, deep atmosphere, Key {key_detected}"

def separate_vocals(input_path):
    """Uses HTDemucs to isolate vocals from the source audio."""
    print(f"Isolating vocals from source: {input_path}")
    clean_path = "temp_input.wav"
    subprocess.call(f'ffmpeg -y -i "{input_path}" -ar 32000 -ac 1 "{clean_path}"', shell=True, stderr=subprocess.DEVNULL)
    subprocess.call(f'python3 -m demucs -n htdemucs --two-stems=vocals "{clean_path}"', shell=True, stdout=subprocess.DEVNULL)

    name = os.path.splitext(clean_path)[0]
    separated_file = f"separated/htdemucs/{name}/vocals.wav"
    return separated_file if os.path.exists(separated_file) else clean_path

def apply_diffsinger_stability(y, sr):
    """
    Applies pitch quantization and stability processing based on DiffSinger acoustic modeling.
    """
    print("Applying F0 pitch stability algorithms...")

    # 1. Pitch Quantization (F0 Flattening)
    try:
        y_tuned = pyrubberband.pitch_shift(y, sr, n_steps=0.0, rbargs=['-F', '-t', '1.0'])
    except:
        y_tuned = y

    # 2. Spectral Processing Chain
    board = Pedalboard([
        HighpassFilter(cutoff_frequency_hz=90),
        Compressor(threshold_db=-20, ratio=4),
        Reverb(room_size=0.4, wet_level=0.3),
        Limiter(threshold_db=-1.0)
    ])
    y_final = board(y_tuned, sr)
    sf.write("vocal_processed.wav", y_final, sr)
    return y_final, sr

def generate_backing_track(vocal_audio, sr, user_vibe):
    """Generates accompaniment using MusicGen based on vocal key and context."""
    print("Initializing Generative Model...")
    device = "cuda" if torch.cuda.is_available() else "cpu"
    torch.cuda.empty_cache()
    gc.collect()

    model = MusicGen.get_pretrained('facebook/musicgen-melody', device=device)

    # Key Detection
    chroma = librosa.feature.chroma_cqt(y=vocal_audio, sr=sr)
    key_idx = np.argmax(np.mean(chroma, axis=1))
    notes = ['C', 'C#', 'D', 'D#', 'E', 'F', 'F#', 'G', 'G#', 'A', 'A#', 'B']
    detected_key = notes[key_idx]

    # Prompt Generation
    smart_prompt = generate_context_prompt(user_vibe, detected_key)

    # Generation Parameters
    duration = min(len(vocal_audio)/sr, 30)
    model.set_generation_params(duration=duration, temperature=0.90, cfg_coef=6.0)

    vocal_tensor = torch.from_numpy(vocal_audio[:int(duration*sr)]).float()
    vocal_tensor = vocal_tensor.unsqueeze(0).unsqueeze(0).to(device)

    print("Generating composition...")
    wav = model.generate_with_chroma(
        descriptions=[smart_prompt],
        melody_wavs=vocal_tensor,
        melody_sample_rate=sr,
        progress=True
    )
    return wav, duration

def mix_audio(vocal, backing, sr):
    """Mixes vocals and backing track using sidechain compression."""
    min_len = min(len(vocal), len(backing))
    v = vocal[:min_len]
    b = backing[:min_len]

    # Dynamic Ducking (Sidechain)
    rms = librosa.feature.rms(y=v)[0]
    rms = librosa.util.fix_length(rms, size=min_len)
    ducking = 1.0 - (rms / (np.max(rms)+1e-9) * 0.4)

    mix = (v * 0.9) + ((b * ducking) * 0.6)
    return np.clip(mix, -0.99, 0.99)

def generate_f0_graph(audio_path, title="F0 Contour Analysis"):
    """Extracts and plots the Fundamental Frequency (F0)."""
    y, sr = librosa.load(audio_path, sr=32000)
    f0, _, _ = librosa.pyin(y, fmin=60, fmax=1000)
    times = librosa.times_like(f0, sr=sr)

    plt.figure(figsize=(10, 4))
    plt.plot(times, f0, color='cyan', linewidth=1.5)
    plt.title(title)
    plt.ylabel("Frequency (Hz)")
    plt.xlabel("Time (s)")
    plt.grid(True, alpha=0.3)

    filename = "f0_analysis_graph.png"
    plt.savefig(filename, dpi=150)
    plt.close()
    return filename

# --- MAIN EXECUTION PIPELINE ---

def main():
    print("--- Audio Processing Pipeline Started ---")

    # 1. File Input
    default_file = "vocal.wav"
    print("Please ensure the audio file is located in the root directory.")
    user_file = input(f"Enter filename (default: '{default_file}'): ").strip()
    if not user_file: user_file = default_file

    if not os.path.exists(user_file):
        print(f"File '{user_file}' not found. Initiating upload...")
        uploaded = files.upload()
        user_file = list(uploaded.keys())[0]

    # 2. Style Input
    user_style = input("Enter Target Style (e.g., Gospel, Pop, Rock): ").strip()

    try:
        # Step A: Separation
        vocal_path = separate_vocals(user_file)

        # Step B: Analysis
        graph_img = generate_f0_graph(vocal_path, title=f"F0 Analysis - {user_style}")

        # Step C: DiffSinger Stability Processing
        y_raw, sr = librosa.load(vocal_path, sr=32000, mono=True)
        vocal_polished, sr = apply_diffsinger_stability(y_raw, sr)

        # Step D: Music Generation
        backing_tracks, duration = generate_backing_track(vocal_polished, sr, user_style)

        # Step E: Final Mix
        bg = backing_tracks[0].cpu().detach().numpy().squeeze()
        final_mix = mix_audio(vocal_polished, bg, sr)

        output_filename = f"Final_Mix_{user_style}.wav"
        sf.write(output_filename, final_mix, sr)

        print("\nProcessing Complete.")
        print("Downloading assets...")
        files.download(output_filename)
        files.download(graph_img)
        files.download("vocal_processed.wav")

    except Exception as e:
        print(f"Execution Error: {e}")

if __name__ == "__main__":
    main()